# Phase 3: Classification
**MIS 637 B — Group 7**  
6 classifiers: Decision Tree, Neural Network, Naïve Bayes, KNN, Logistic Regression, Random Forest

In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score)
from imblearn.over_sampling import SMOTE

sns.set_theme(style='whitegrid')

X = np.load('../data/X_scaled.npy')
y = np.load('../data/y.npy')

## 1. Train/Test Split + SMOTE (handle class imbalance)

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print('After SMOTE:', np.bincount(y_train_res))

After SMOTE: [42092 42092]


## 2. Define Models

In [3]:
models = {
    'Decision Tree (C4.5)': DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=42),
    'Neural Network (BP)':  MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, random_state=42),
    'Naïve Bayes':          GaussianNB(),
    'KNN':                  KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

## 3. Train + Evaluate All Models

In [4]:
results = []
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_res, y_train_res)
    trained_models[name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    results.append({
        'Model': name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan,
    })

results_df = pd.DataFrame(results).set_index('Model').round(4)
results_df

Training Decision Tree (C4.5)...


Training Neural Network (BP)...


Training Naïve Bayes...
Training KNN...


Training Logistic Regression...
Training Random Forest...


/opt/homebrew/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,
Decision Tree (C4.5),0.8778,0.3277,0.1036,0.1574,0.6233
Neural Network (BP),0.7722,0.1486,0.2256,0.1792,0.5552
Naïve Bayes,0.1139,0.1101,0.9946,0.1983,0.5886
KNN,0.6326,0.1368,0.4398,0.2087,0.5609
Logistic Regression,0.6487,0.1628,0.5280,0.2488,0.6290
Random Forest,0.8883,0.4118,0.0322,0.0598,0.6250


In [5]:
results_df.to_csv('../outputs/model_comparison.csv')

# Save all models
with open('../outputs/models/all_models.pkl', 'wb') as f:
    pickle.dump(trained_models, f)

print('Saved.')

Saved.
